<a href="https://colab.research.google.com/github/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/blob/main/notebooks/01_exploracion_dfire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import platform
import sys

print("Python:", sys.version)
print("Sistema:", platform.system())
print("Directorio actual:", os.getcwd())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sistema: Linux
Directorio actual: /content


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó una GPU.")

PyTorch: 2.11.0+cu128
GPU disponible: True
GPU: Tesla T4


In [4]:
!pip install -q -r https://raw.githubusercontent.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego/main/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.4 MB/s eta 0:00:00


In [5]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
from pathlib import Path

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego.git"
REPO_DIR = Path("/content/VpC2---Deteccion-de-humo-y-fuego")

if not REPO_DIR.exists():
    !git clone {REPO_URL}
else:
    print("El repositorio ya está clonado.")

%cd {REPO_DIR}

Cloning into 'VpC2---Deteccion-de-humo-y-fuego'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 21 (delta 6), reused 10 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 2.25 MiB | 33.40 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/VpC2---Deteccion-de-humo-y-fuego


In [7]:
!pip install -q -r requirements.txt

In [8]:
import kagglehub

DATA_DIR = Path("/content/data/dfire")
DATA_DIR.mkdir(parents=True, exist_ok=True)

dataset_path = Path(
    kagglehub.dataset_download(
        "sayedgamal99/smoke-fire-detection-yolo",
        output_dir=str(DATA_DIR)
    )
)

print("Dataset descargado en:")
print(dataset_path)

Using Colab cache for faster access to the 'smoke-fire-detection-yolo' dataset.
Dataset descargado en:
/kaggle/input/smoke-fire-detection-yolo


In [9]:
def mostrar_estructura(root: Path, max_depth: int = 3) -> None:
    """Muestra la estructura de directorios sin imprimir miles de archivos."""

    root = root.resolve()
    print(root.name)

    for path in sorted(root.rglob("*")):
        relative_path = path.relative_to(root)

        if len(relative_path.parts) > max_depth:
            continue

        indent = "    " * len(relative_path.parts)
        icon = "📁" if path.is_dir() else "📄"

        print(f"{indent}{icon} {path.name}")


mostrar_estructura(dataset_path, max_depth=3)

smoke-fire-detection-yolo
    📁 data
        📁 test
            📁 images
            📁 labels
        📁 train
            📁 images
            📁 labels
        📁 val
            📁 images
            📁 labels
    📄 data.yaml


In [10]:
import yaml
yaml_paths = list(dataset_path.rglob("*.yaml"))

if not yaml_paths:
    print("No se encontraron archivos YAML.")
else:
    for yaml_path in yaml_paths:
        print("=" * 60)
        print(f"Archivo: {yaml_path}")

        with open(yaml_path, "r", encoding="utf-8") as file:
            yaml_content = yaml.safe_load(file)

        print(yaml.dump(
            yaml_content,
            allow_unicode=True,
            sort_keys=False
        ))

Archivo: /kaggle/input/smoke-fire-detection-yolo/data.yaml
path: /kaggle/working/D Fire Dataset
train: data/train/images
val: data/val/images
test: data/test/images
names:
- smoke
- fire
nc: 2
train_count: 14122
val_count: 3099
test_count: 4306



In [11]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def contar_archivos_split(split_path: Path) -> dict:
    split_path = Path(split_path)

    images_dir = split_path / "images"
    labels_dir = split_path / "labels"

    images = (
        [
            path
            for path in images_dir.rglob("*")
            if path.suffix.lower() in IMAGE_EXTENSIONS
        ]
        if images_dir.exists()
        else []
    )

    labels = (
        list(labels_dir.rglob("*.txt"))
        if labels_dir.exists()
        else []
    )

    return {
        "split": split_path.name,
        "imagenes": len(images),
        "etiquetas": len(labels),
        "images_dir": str(images_dir),
        "labels_dir": str(labels_dir),
    }

In [12]:
label_paths = [
    path
    for path in dataset_path.rglob("*.txt")
    if "labels" in [part.lower() for part in path.parts]
]

print(f"Cantidad total de imágenes: {len(image_paths)}")
print(f"Cantidad de archivos dentro de carpetas labels: {len(label_paths)}")

NameError: name 'image_paths' is not defined

In [ ]:
from pathlib import Path

DATASET_ROOT = Path("/content/data/dfire")
DATA_DIR = DATASET_ROOT / "data"

for split in ["train", "val", "test"]:
    images_dir = DATA_DIR / split / "images"
    labels_dir = DATA_DIR / split / "labels"

    print(f"\nSplit: {split}")
    print(f"Imágenes: {images_dir}")
    print(f"Existe: {images_dir.exists()}")
    print(f"Etiquetas: {labels_dir}")
    print(f"Existe: {labels_dir.exists()}")

In [ ]:
import yaml

dfire_config = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {
        0: "smoke",
        1: "fire",
    },
    "nc": 2,
}

DFIRE_YAML = Path("/content/dfire_colab.yaml")

with open(DFIRE_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        dfire_config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print(f"Configuración creada en: {DFIRE_YAML}")

with open(DFIRE_YAML, "r", encoding="utf-8") as file:
    print(file.read())

In [ ]:
import pandas as pd

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

resultados = []

for split in ["train", "val", "test"]:
    images_dir = DATA_DIR / split / "images"
    labels_dir = DATA_DIR / split / "labels"

    images = [
        path
        for path in images_dir.iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(labels_dir.glob("*.txt"))

    image_stems = {path.stem for path in images}
    label_stems = {path.stem for path in labels}

    resultados.append({
        "split": split,
        "imagenes": len(images),
        "etiquetas": len(labels),
        "imagenes_sin_etiqueta": len(image_stems - label_stems),
        "etiquetas_sin_imagen": len(label_stems - image_stems),
        "etiquetas_vacias": sum(
            not path.read_text(encoding="utf-8").strip()
            for path in labels
        ),
    })

df_archivos = pd.DataFrame(resultados)
df_archivos

In [ ]:
from collections import Counter

CLASS_NAMES = {
    0: "smoke",
    1: "fire",
}

distribucion_splits = []
errores_anotacion = []

for split in ["train", "val", "test"]:
    labels_dir = DATA_DIR / split / "labels"
    class_counter = Counter()

    for label_path in labels_dir.glob("*.txt"):
        contenido = label_path.read_text(encoding="utf-8").strip()

        if not contenido:
            continue

        for numero_linea, linea in enumerate(contenido.splitlines(), start=1):
            valores = linea.split()

            if len(valores) != 5:
                errores_anotacion.append({
                    "archivo": str(label_path),
                    "linea": numero_linea,
                    "problema": f"Se esperaban 5 valores y se encontraron {len(valores)}",
                })
                continue

            class_id = int(valores[0])
            coordinates = list(map(float, valores[1:]))

            if class_id not in CLASS_NAMES:
                errores_anotacion.append({
                    "archivo": str(label_path),
                    "linea": numero_linea,
                    "problema": f"Clase desconocida: {class_id}",
                })
                continue

            if not all(0 <= value <= 1 for value in coordinates):
                errores_anotacion.append({
                    "archivo": str(label_path),
                    "linea": numero_linea,
                    "problema": "Coordenadas fuera del rango [0, 1]",
                })
                continue

            class_counter[class_id] += 1

    distribucion_splits.append({
        "split": split,
        "smoke_boxes": class_counter[0],
        "fire_boxes": class_counter[1],
        "total_boxes": sum(class_counter.values()),
    })

df_distribucion = pd.DataFrame(distribucion_splits)
df_distribucion

In [ ]:
print(f"Cantidad de anotaciones problemáticas: {len(errores_anotacion)}")

if errores_anotacion:
    display(pd.DataFrame(errores_anotacion).head(20))
else:
    print("No se encontraron errores básicos en las anotaciones.")

In [ ]:
def obtener_tipo_imagen(label_path: Path) -> str:
    """
    Clasifica una imagen según las clases presentes
    en su archivo de etiquetas YOLO.
    """
    contenido = label_path.read_text(encoding="utf-8").strip()

    if not contenido:
        return "negative"

    clases = {
        int(linea.split()[0])
        for linea in contenido.splitlines()
        if linea.strip()
    }

    if clases == {0}:
        return "smoke"

    if clases == {1}:
        return "fire"

    if clases == {0, 1}:
        return "smoke_and_fire"

    return "unknown"

In [ ]:
registros_imagenes = []

for split in ["train", "val", "test"]:
    images_dir = DATA_DIR / split / "images"
    labels_dir = DATA_DIR / split / "labels"

    for image_path in images_dir.iterdir():
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        label_path = labels_dir / f"{image_path.stem}.txt"

        registros_imagenes.append({
            "split": split,
            "image_path": image_path,
            "label_path": label_path,
            "tipo": obtener_tipo_imagen(label_path),
        })

df_imagenes = pd.DataFrame(registros_imagenes)

print(f"Imágenes analizadas: {len(df_imagenes)}")
df_imagenes.head()

In [ ]:
df_tipos = (
    df_imagenes
    .groupby(["split", "tipo"])
    .size()
    .unstack(fill_value=0)
)

df_tipos

In [ ]:
df_tipos_porcentaje = (
    df_tipos
    .div(df_tipos.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

df_tipos_porcentaje

In [ ]:
import cv2
import matplotlib.pyplot as plt


CLASS_NAMES = {
    0: "smoke",
    1: "fire",
}

CLASS_COLORS = {
    0: (128, 128, 128),  # gris para humo
    1: (255, 0, 0),      # rojo para fuego
}


def leer_etiquetas_yolo(label_path: Path) -> list:
    """Lee las anotaciones de un archivo YOLO."""
    contenido = label_path.read_text(encoding="utf-8").strip()

    if not contenido:
        return []

    anotaciones = []

    for linea in contenido.splitlines():
        valores = linea.split()

        class_id = int(valores[0])
        x_center, y_center, box_width, box_height = map(
            float,
            valores[1:]
        )

        anotaciones.append({
            "class_id": class_id,
            "x_center": x_center,
            "y_center": y_center,
            "width": box_width,
            "height": box_height,
        })

    return anotaciones

In [ ]:
def dibujar_anotaciones(
    image_path: Path,
    label_path: Path,
):
    """Dibuja las bounding boxes YOLO sobre una imagen."""

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        raise ValueError(f"No se pudo leer la imagen: {image_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    image_height, image_width = image_rgb.shape[:2]
    anotaciones = leer_etiquetas_yolo(label_path)

    for anotacion in anotaciones:
        class_id = anotacion["class_id"]

        x_center = anotacion["x_center"] * image_width
        y_center = anotacion["y_center"] * image_height
        box_width = anotacion["width"] * image_width
        box_height = anotacion["height"] * image_height

        x_min = int(x_center - box_width / 2)
        y_min = int(y_center - box_height / 2)
        x_max = int(x_center + box_width / 2)
        y_max = int(y_center + box_height / 2)

        x_min = max(0, x_min)
        y_min = max(0, y_min)
        x_max = min(image_width - 1, x_max)
        y_max = min(image_height - 1, y_max)

        color = CLASS_COLORS[class_id]
        class_name = CLASS_NAMES[class_id]

        cv2.rectangle(
            image_rgb,
            (x_min, y_min),
            (x_max, y_max),
            color,
            thickness=3,
        )

        cv2.putText(
            image_rgb,
            class_name,
            (x_min, max(y_min - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            thickness=2,
        )

    return image_rgb

In [ ]:
TIPOS = [
    "negative",
    "smoke",
    "fire",
    "smoke_and_fire",
]

muestras = []

for tipo in TIPOS:
    candidatos = df_imagenes[
        (df_imagenes["split"] == "train")
        & (df_imagenes["tipo"] == tipo)
    ]

    cantidad = min(2, len(candidatos))

    if cantidad > 0:
        muestras_tipo = candidatos.sample(
            n=cantidad,
            random_state=42,
        )

        muestras.append(muestras_tipo)

df_muestras = pd.concat(muestras, ignore_index=True)

df_muestras[["split", "tipo", "image_path"]]

In [ ]:
fig, axes = plt.subplots(
    nrows=4,
    ncols=2,
    figsize=(14, 18),
)

axes = axes.flatten()

for axis, (_, fila) in zip(axes, df_muestras.iterrows()):
    image = dibujar_anotaciones(
        fila["image_path"],
        fila["label_path"],
    )

    axis.imshow(image)
    axis.set_title(fila["tipo"])
    axis.axis("off")

for axis in axes[len(df_muestras):]:
    axis.axis("off")

plt.tight_layout()
plt.show()